In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("../data/processed/model_ready_data.csv")

neighbourhood = pd.read_csv("../data/processed/neighbourhood_demand.csv")
neighbourhood["datetime"] = pd.to_datetime(neighbourhood["datetime"])

df.head()

,datetime,total_demand_kwh,hour,day,month,weekday,is_weekend,season,lag_1,lag_48,lag_96,rolling_mean_48,rolling_std_48
0,2013-01-03 00:00:00,209.864,0,3,1,3,0,0,243.522,198.338,220.013,240.736854,91.920733
1,2013-01-03 00:30:00,175.020,0,3,1,3,0,0,209.864,165.862,204.270,240.927646,91.771399
2,2013-01-03 01:00:00,151.394,1,3,1,3,0,0,175.020,147.971,189.439,240.998958,91.698930
3,2013-01-03 01:30:00,137.672,1,3,1,3,0,0,151.394,129.889,173.019,241.161104,91.504956
4,2013-01-03 02:00:00,128.217,2,3,1,3,0,0,137.672,117.542,154.537,241.383500,91.210616


MAPE

In [3]:
features = [
    "hour", "day", "month", "weekday", "is_weekend", "season",
    "lag_1", "lag_48", "lag_96",
    "rolling_mean_48", "rolling_std_48"
]

X = df[features]
y = df["total_demand_kwh"]

split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [4]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

In [5]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

model_metrics = pd.DataFrame({
    "Model": ["XGBoost"],
    "MAE": [mae],
    "RMSE": [rmse],
    "MAPE": [mape],
    "R2": [r2]
})

model_metrics

,Model,MAE,RMSE,MAPE,R2
0,XGBoost,5.707997,7.632833,2.578478,0.991178


Mean + 2σ threshold

In [6]:
neighbourhood = neighbourhood[
    neighbourhood["datetime"] >= "2013-01-01"
].copy()

stress_threshold_95 = neighbourhood["total_demand_kwh"].quantile(0.95)

mean_demand = neighbourhood["total_demand_kwh"].mean()
std_demand = neighbourhood["total_demand_kwh"].std()

threshold_2sd = mean_demand + (2 * std_demand)

print("95th Percentile Threshold:", stress_threshold_95)
print("Mean + 2SD Threshold:", threshold_2sd)

95th Percentile Threshold: 354.5756
Mean + 2SD Threshold: 358.7396694672242


In [7]:
stress_events_95 = (
    neighbourhood["total_demand_kwh"] > stress_threshold_95
).sum()

stress_percentage_95 = (
    stress_events_95 / len(neighbourhood)
) * 100

stress_events_2sd = (
    neighbourhood["total_demand_kwh"] > threshold_2sd
).sum()

stress_percentage_2sd = (
    stress_events_2sd / len(neighbourhood)
) * 100

In [8]:
threshold_comparison = pd.DataFrame({
    "Threshold_Method": [
        "95th Percentile",
        "Mean + 2SD"
    ],
    "Threshold_Value": [
        stress_threshold_95,
        threshold_2sd
    ],
    "Stress_Events": [
        stress_events_95,
        stress_events_2sd
    ],
    "Stress_Percentage": [
        stress_percentage_95,
        stress_percentage_2sd
    ]
})

threshold_comparison

,Threshold_Method,Threshold_Value,Stress_Events,Stress_Percentage
0,95th Percentile,354.575600,1016,5.003694
1,Mean + 2SD,358.739669,921,4.535829


Ramp Rate feature

In [9]:
df_ramp = df.copy()

df_ramp["ramp_rate"] = (
    df_ramp["total_demand_kwh"].diff()
)

df_ramp = df_ramp.dropna().copy()

df_ramp.head()

,datetime,total_demand_kwh,hour,day,month,weekday,is_weekend,season,lag_1,lag_48,lag_96,rolling_mean_48,rolling_std_48,ramp_rate
1,2013-01-03 00:30:00,175.020,0,3,1,3,0,0,209.864,165.862,204.270,240.927646,91.771399,-34.844
2,2013-01-03 01:00:00,151.394,1,3,1,3,0,0,175.020,147.971,189.439,240.998958,91.698930,-23.626
3,2013-01-03 01:30:00,137.672,1,3,1,3,0,0,151.394,129.889,173.019,241.161104,91.504956,-13.722
4,2013-01-03 02:00:00,128.217,2,3,1,3,0,0,137.672,117.542,154.537,241.383500,91.210616,-9.455
5,2013-01-03 02:30:00,121.280,2,3,1,3,0,0,128.217,113.493,147.255,241.545729,90.984953,-6.937


In [10]:
features_ramp = features + ["ramp_rate"]

X_ramp = df_ramp[features_ramp]
y_ramp = df_ramp["total_demand_kwh"]

split_index = int(len(df_ramp) * 0.8)

X_train_ramp = X_ramp.iloc[:split_index]
X_test_ramp = X_ramp.iloc[split_index:]

y_train_ramp = y_ramp.iloc[:split_index]
y_test_ramp = y_ramp.iloc[split_index:]

In [11]:
xgb_ramp = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_ramp.fit(X_train_ramp, y_train_ramp)

y_pred_ramp = xgb_ramp.predict(X_test_ramp)

In [12]:
ramp_metrics = pd.DataFrame({
    "Model": ["XGBoost with Ramp Rate"],
    "MAE": [mean_absolute_error(y_test_ramp, y_pred_ramp)],
    "RMSE": [np.sqrt(mean_squared_error(y_test_ramp, y_pred_ramp))],
    "MAPE": [mean_absolute_percentage_error(y_test_ramp, y_pred_ramp) * 100],
    "R2": [r2_score(y_test_ramp, y_pred_ramp)]
})

ramp_metrics

,Model,MAE,RMSE,MAPE,R2
0,XGBoost with Ramp Rate,1.798479,2.493022,0.851864,0.999059


Time Series Cross Validation

In [13]:
tscv = TimeSeriesSplit(n_splits=5)

cv_results = []

for fold, (train_index, test_index) in enumerate(tscv.split(X), start=1):

    X_train_cv = X.iloc[train_index]
    X_test_cv = X.iloc[test_index]

    y_train_cv = y.iloc[train_index]
    y_test_cv = y.iloc[test_index]

    model = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train_cv, y_train_cv)

    y_pred_cv = model.predict(X_test_cv)

    cv_results.append({
        "Fold": fold,
        "MAE": mean_absolute_error(y_test_cv, y_pred_cv),
        "RMSE": np.sqrt(mean_squared_error(y_test_cv, y_pred_cv)),
        "MAPE": mean_absolute_percentage_error(y_test_cv, y_pred_cv) * 100,
        "R2": r2_score(y_test_cv, y_pred_cv)
    })

cv_results = pd.DataFrame(cv_results)

cv_results

,Fold,MAE,RMSE,MAPE,R2
0,1,13.011520,16.562033,7.878802,0.944248
1,2,5.490081,7.363084,3.261564,0.973328
2,3,4.930855,6.856900,2.857193,0.983003
3,4,6.324970,8.607243,3.012800,0.987774
4,5,5.734340,7.666435,2.597769,0.991067


In [14]:
cv_summary = cv_results.mean(numeric_only=True)

cv_summary

Fold    3.000000
MAE     7.098353
RMSE    9.411139
MAPE    3.921626
R2      0.975884
dtype: float64

Grid Search

In [15]:
param_grid = {
    "max_depth": [3, 5, 6],
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0]
}

In [16]:
grid_model = XGBRegressor(
    random_state=42,
    colsample_bytree=0.8
)

grid_search = GridSearchCV(
    estimator=grid_model,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 54 candidates, totalling 162 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.03, 0.05, ...], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200, ...], 'subsample': [0.8, 1.0]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for

In [17]:
print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV RMSE:")
print(-grid_search.best_score_)

Best Parameters:
{'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 300, 'subsample': 1.0}
Best CV RMSE:
9.436720413646556


In [18]:
best_xgb = grid_search.best_estimator_

y_pred_best = best_xgb.predict(X_test)

grid_metrics = pd.DataFrame({
    "Model": ["Grid Search XGBoost"],
    "MAE": [mean_absolute_error(y_test, y_pred_best)],
    "RMSE": [np.sqrt(mean_squared_error(y_test, y_pred_best))],
    "MAPE": [mean_absolute_percentage_error(y_test, y_pred_best) * 100],
    "R2": [r2_score(y_test, y_pred_best)]
})

grid_metrics

,Model,MAE,RMSE,MAPE,R2
0,Grid Search XGBoost,5.808036,7.757396,2.627608,0.990888


Improved Sensitivity Analysis

In [19]:
neighbourhood["hour"] = neighbourhood["datetime"].dt.hour
neighbourhood["month"] = neighbourhood["datetime"].dt.month

neighbourhood["ev_load"] = 0
neighbourhood.loc[
    neighbourhood["hour"].between(17, 21),
    "ev_load"
] = 1

neighbourhood["is_winter"] = (
    neighbourhood["month"].isin([12, 1, 2])
).astype(int)

neighbourhood["heating_period"] = (
    neighbourhood["hour"].between(6, 9)
    |
    neighbourhood["hour"].between(17, 22)
).astype(int)

neighbourhood["hp_load"] = (
    neighbourhood["is_winter"]
    *
    neighbourhood["heating_period"]
)

In [20]:
stress_threshold = 354.5756

EV sensitivity

In [21]:
ev_sensitivity = []

for ev_load in [10, 20, 30, 40, 50, 60, 70, 80]:

    future_demand = (
        neighbourhood["total_demand_kwh"]
        +
        neighbourhood["ev_load"] * ev_load
    )

    stress_events = (future_demand > stress_threshold).sum()
    stress_percentage = (stress_events / len(neighbourhood)) * 100

    ev_sensitivity.append({
        "EV_Load": ev_load,
        "Stress_Events": stress_events,
        "Stress_Percentage": stress_percentage
    })

ev_sensitivity = pd.DataFrame(ev_sensitivity)

ev_sensitivity

,EV_Load,Stress_Events,Stress_Percentage
0,10,1252,6.165969
1,20,1475,7.264221
2,30,1691,8.327998
3,40,1889,9.303127
4,50,2076,10.224083
5,60,2249,11.076090
6,70,2390,11.770500
7,80,2547,12.543708


Heat pump sensitivity

In [22]:
hp_sensitivity = []

for hp_load in [5, 10, 15, 20, 25, 30, 35, 40]:

    future_demand = (
        neighbourhood["total_demand_kwh"]
        +
        neighbourhood["hp_load"] * hp_load
    )

    stress_events = (future_demand > stress_threshold).sum()
    stress_percentage = (stress_events / len(neighbourhood)) * 100

    hp_sensitivity.append({
        "HP_Load": hp_load,
        "Stress_Events": stress_events,
        "Stress_Percentage": stress_percentage
    })

hp_sensitivity = pd.DataFrame(hp_sensitivity)

hp_sensitivity

,HP_Load,Stress_Events,Stress_Percentage
0,5,1096,5.397685
1,10,1181,5.816301
2,15,1272,6.264467
3,20,1347,6.633834
4,25,1416,6.973652
5,30,1500,7.387343
6,35,1569,7.727161
7,40,1631,8.032504


Interaction Analysis

In [23]:
baseline_stress = 5.0037

ev_extreme_stress = 12.54
hp_extreme_stress = 8.03
combined_extreme_stress = 13.0116

expected_combined = (
    ev_extreme_stress
    +
    hp_extreme_stress
    -
    baseline_stress
)

interaction_excess = (
    combined_extreme_stress
    -
    expected_combined
)

print("Expected Combined Stress:", expected_combined)
print("Actual Combined Stress:", combined_extreme_stress)
print("Interaction Excess:", interaction_excess)

Expected Combined Stress: 15.5663
Actual Combined Stress: 13.0116
Interaction Excess: -2.5547000000000004


In [24]:
if interaction_excess > 0:
    interaction_type = "Super-additive"
elif interaction_excess < 0:
    interaction_type = "Sub-additive"
else:
    interaction_type = "Additive"

interaction_type

'Sub-additive'

In [25]:
interaction_results = pd.DataFrame({
    "Metric": [
        "Baseline Stress %",
        "EV Extreme Stress %",
        "HP Extreme Stress %",
        "Expected Combined Stress %",
        "Actual Combined Stress %",
        "Interaction Excess",
        "Interaction Type"
    ],
    "Value": [
        baseline_stress,
        ev_extreme_stress,
        hp_extreme_stress,
        expected_combined,
        combined_extreme_stress,
        interaction_excess,
        interaction_type
    ]
})

interaction_results

,Metric,Value
0,Baseline Stress %,5.0037
1,EV Extreme Stress %,12.54
2,HP Extreme Stress %,8.03
3,Expected Combined Stress %,15.5663
4,Actual Combined Stress %,13.0116
5,Interaction Excess,-2.5547
6,Interaction Type,Sub-additive


Carbon Intensity API

In [26]:
import requests

url = "https://api.carbonintensity.org.uk/intensity"

response = requests.get(url)

data = response.json()

data

{'data': [{'from': '2026-06-27T21:00Z',
   'to': '2026-06-27T21:30Z',
   'intensity': {'forecast': 174, 'actual': 181, 'index': 'high'}}]}

In [27]:
current_intensity = data["data"][0]["intensity"]["actual"]
forecast_intensity = data["data"][0]["intensity"]["forecast"]
index = data["data"][0]["intensity"]["index"]

print("Current Carbon Intensity:", current_intensity)
print("Forecast Carbon Intensity:", forecast_intensity)
print("Index:", index)

Current Carbon Intensity: 181
Forecast Carbon Intensity: 174
Index: high


In [28]:
carbon_value = current_intensity

if carbon_value is None:
    carbon_value = forecast_intensity

carbon_value

181